# 2f-theo — Koszul Jacobi $\Leftrightarrow [\pi,\pi]_{SN}=0$ (theorem citation)

**Problem (f).** $T^*M$ üzerindeki Koszul bracket'in Jacobi özdeşliği:

$$
\sum_{\text{cyc}}\bigl[\alpha,[\beta,\gamma]_{T^*M}\bigr]_{T^*M} = 0
\qquad \Longleftrightarrow \qquad [\pi,\pi]_{SN} = 0.
$$

Yani: $T^*M$ Koszul bracket'i Lie cebri yasasını sağlar **eğer ve ancak** $\pi$ bivector'unun Schouten-Nijenhuis öz-bracket'i sıfırsa (yani $\pi$ gerçek bir Poisson tensörüyse).

**Bu notebook'un yaklaşımı (2f-theo).** Paketin Faz 9 Stage B.3'te seeded etmiş olduğu `poisson_koszul_jacobi` teoremini doğrudan cite ediyoruz: tek-adım `DerivedBracketTheorem` kuralıyla cyclic Jacobi sum'ı $[\pi,\pi]_{SN}$'e indirir.

**Karşılaştırma (2f-deep, ayrı notebook).** Aynı sonucu **teoremi sıfırdan ispatlayarak** elde etmek — 27-terim açılımı + 6 inline aksiyom + ~100 adımlık zincir. O pass Faz 13 kapsamında ayrılmıştır; bu notebook'tan sonra yazılır.

## Strateji — niye theorem cite

Bu özdeşlik **Derived Bracket Theorem'in** standart bir corollary'si: bir derived bracket $\{\cdot,\cdot\}_Q := [[\cdot, Q], \cdot]$ üzerinde graded Jacobi $\Leftrightarrow [Q,Q]_{\text{base}}=0$. Koszul bracket bunun $Q=\pi$, base bracket $= [\cdot,\cdot]_{SN}$, anchor $= \pi^\sharp$ özel hâlidir.

Paket bu teoremi `THEOREM_POISSON_KOSZUL_JACOBI` olarak `theorem_book`'ta taşıyor. `PoissonBracket.prove_koszul_jacobi_reduction(α, β, γ)` tek satırla bu teoremi instantiate eder ve `ProofChain` üretir.

**Iki yön:**

- **(⟸)** Eğer $[\pi,\pi]_{SN}=0$ ise, teoremin RHS'i sıfırlanır, dolayısıyla sol taraf — cyclic Koszul Jacobi sum — sıfırdır.
- **(⟹)** Eğer cyclic sum tüm $(\alpha,\beta,\gamma)$ için sıfırsa, teorem RHS'inin de sıfır olması gerekir — yani $[\pi,\pi]_{SN}$ universal obstruction olarak vanish eder.

İkinci yön, teoremi **iff** olarak okumayı gerektirir. Paketin 1-step zinciri *aşağı yönü* (LHS → RHS) verir; iff'in iki yönü de aynı `BracketApply([·,·]_SN, π, π)` obstruction'ından çıkar.

In [1]:
# Notebook doğrudan açıldığında jacopy'ı import edilebilir hâle getirir.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

from jacopy.core.expr import Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.library.poisson import PoissonBracket
from jacopy.library.theorem_book import theorem_book

## 1. Kurulum

- $\pi$ — bivector (SN-derecesi 1).
- $\alpha, \beta, \gamma$ — generic 1-formlar.
- `PoissonBracket.from_bivector(π)` — Koszul + Poisson view'lerini birlikte taşıyan library wrapper.

In [2]:
reg = PropertyRegistry()

pi = Symbol("π")
alpha = Symbol("α")
beta = Symbol("β")
gamma = Symbol("γ")

reg.declare(pi, Graded(degree=1))
reg.declare(alpha, Graded(degree=1))
reg.declare(beta, Graded(degree=1))
reg.declare(gamma, Graded(degree=1))

poisson = PoissonBracket.from_bivector(pi)

print(f"π            : {pi}  (bivector)")
print(f"α, β, γ      : {alpha} {beta} {gamma}")
print(f"PoissonBracket : {poisson}")

π            : π  (bivector)
α, β, γ      : α β γ
PoissonBracket : PoissonBracket(π=π)


## 2. Cyclic Koszul Jacobi obstruction

Wrapper'ın `koszul_jacobi_condition()` metodu, cyclic sum'ın hangi `Expr`'e indirgendiğini söyler:

In [3]:
cond = poisson.koszul_jacobi_condition(registry=reg)
print(f"condition name : {cond.name}")
print(f"obstruction    : {cond.obstruction}")

condition name : Koszul Jacobi condition on {·,·}_π
obstruction    : [·,·]_SN(π, π)


**Yorum.** Vanishing condition `[·,·]_SN(π, π)` — yani cyclic Koszul Jacobi sum'ın evrensel obstruction'ı *tek bir* expression: $[\pi,\pi]_{SN}$. Bu, $(\alpha,\beta,\gamma)$ üçlüsünden bağımsız — Derived Bracket Theorem'in ana içeriği bu evrenselliktir.

## 3. İspat zinciri — `prove_koszul_jacobi_reduction`

`PoissonBracket` üzerindeki `prove_koszul_jacobi_reduction(α, β, γ)` metodu, cyclic Jacobi sum → $[\pi,\pi]_{SN}$ indirgemesini bir `ProofChain` olarak üretir.

In [4]:
chain = poisson.prove_koszul_jacobi_reduction(alpha, beta, gamma, registry=reg)

print(f"KAPANDI — {len(chain)} adım.\n")
for i, step in enumerate(chain.steps, 1):
    print(f"[{i}] rule = {step.rule}")
    print(f"    just = {step.justification}")
    print(f"    LHS  : {step.before}")
    print(f"    RHS  : {step.after}")

KAPANDI — 1 adım.

[1] rule = DerivedBracketTheorem
    just = Jacobi on {·,·}_π,♯ ⟺ [π, π]_SN = 0 (Derived Bracket Theorem)
    LHS  : ((-{·,·}_π,♯(α, {·,·}_π,♯(β, γ))) + (-{·,·}_π,♯(β, {·,·}_π,♯(γ, α))) + (-{·,·}_π,♯(γ, {·,·}_π,♯(α, β))))
    RHS  : [·,·]_SN(π, π)


## 4. LaTeX gösterimi

In [5]:
from jacopy.display.jupyter import display_chain
display_chain(chain)

\begin{align*}
-\left[\alpha,\, \left[\beta,\, \gamma\right]_{{\cdot,\cdot}_\pi,\sharp}\right]_{{\cdot,\cdot}_\pi,\sharp} - \left[\beta,\, \left[\gamma,\, \alpha\right]_{{\cdot,\cdot}_\pi,\sharp}\right]_{{\cdot,\cdot}_\pi,\sharp} - \left[\gamma,\, \left[\alpha,\, \beta\right]_{{\cdot,\cdot}_\pi,\sharp}\right]_{{\cdot,\cdot}_\pi,\sharp} &\to \left[\pi,\, \pi\right]_{[\cdot,\cdot]_{SN}} && \text{[DerivedBracketTheorem]\,(theorem)}\;\text{--- Jacobi on {\ensuremath{\cdot},\ensuremath{\cdot}}\_\ensuremath{\pi},\ensuremath{\sharp} ⟺ [\ensuremath{\pi}, \ensuremath{\pi}]\_SN = 0 (Derived Bracket Theorem)}
\end{align*}

## 5. Theorem book kaydı

Bu sonuç paketin teorem kütüphanesinde `poisson_koszul_jacobi` ismiyle kayıtlı (Faz 9 Stage B.3'te seeded edildi).

In [6]:
thm = theorem_book.get("poisson_koszul_jacobi")
print(f"name      : {thm.name}")
print(f"statement : {thm.statement}")
print(f"\nfrom_axioms:")
for ax in thm.from_axioms:
    print(f"  - {ax}")
print(f"\nnotes:\n  {thm.notes}")

name      : poisson_koszul_jacobi
statement : Koszul Jacobi on {·,·}_π cyclic sum = 0 when [π, π]_SN = 0

from_axioms:
  - Derived Bracket Theorem
  - π^♯ = Sharp(π) as form-lift anchor
  - [π, π]_SN = 0 (Poisson hypothesis)

notes:
  Form-level analogue of poisson_jacobi. The Koszul view's Jacobi obstruction coincides with the SN self-bracket [π, π]_SN — anchor ``acting_on=Sharp(π)`` reshapes the expansion but leaves [Q, Q]_base untouched — so one Poisson hypothesis discharges both views at once.


## 6. Iff'in iki yönü

Yukarıdaki tek-adım zincir şunu söyler:

$$
\text{cyclic Jacobi sum on } (\alpha,\beta,\gamma) \;=\; [\pi,\pi]_{SN}\text{-pairing-on-}(\alpha,\beta,\gamma).
$$

Bu eşitlik **identity** — herhangi bir hipotez gerektirmez. Iff iki yönü buradan tek satırda çıkar:

**(⟸)** Hipotez: $[\pi,\pi]_{SN} = 0$. Bu varsayımla zincirin sağ tarafı sıfırdır, dolayısıyla **her** $(\alpha,\beta,\gamma)$ üçlüsü için cyclic sum sıfırlanır — yani Koszul bracket Jacobi'yi sağlar.

**(⟹)** Hipotez: cyclic sum **her** $(\alpha,\beta,\gamma)$ için sıfır. Zincir tarafından bu, $[\pi,\pi]_{SN}$'in tüm 1-form üçlülerine pair edildiğinde sıfır olduğu anlamına gelir. Pairing non-degenerate olduğu için (1-form modülü ayırıcıdır 3-vector field'leri), bu yalnızca $[\pi,\pi]_{SN}=0$ ile mümkündür.

İkinci yön'ün son adımı ("non-degenerate pairing $\Rightarrow$ vanishing tensor") — paketin **bugün engine'inde** ifade edilemez (Faz 12 #9 deferral'ında not edilen `injectivity dispatch` paterni). Pratikte bu aşama matematiksel argüman olarak transcript'e eklenir; tek-adım zincir kendi iddiasını LHS$\equiv$RHS olarak tutar.

## Sonuç

$$
\boxed{\;\sum_{\text{cyc}}\bigl[\alpha,[\beta,\gamma]_K\bigr]_K = 0 \;\Longleftrightarrow\; [\pi,\pi]_{SN}=0.\;}
$$

**1-adımlı zincir, 0 inline aksiyom.** Bu pass'in farkı şu: 2a-2e el-yazımı `Definition` blokları kuruyordu; 2f-theo paketin **mevcut teorem kütüphanesini** kullanıyor. Paketin Faz 9'da yatırım yaptığı tüm SN/Derived-Bracket altyapısı tek satırlık callsite'a indirgenmiş.

**`prove_koszul_jacobi_reduction` ne yapıyor (tek adım altında):**

1. `DerivedBracket(base=SN, Q=π, acting_on=π^♯)` üzerinden Koszul view'ini kurar.
2. `graded_jacobi_obstruction(α, β, γ)` cyclic sum'ı inşa eder (LHS).
3. `jacobi_obstruction_raw()` $[\pi,\pi]_{SN}$ raw `BracketApply` node'unu inşa eder (RHS).
4. **Derived Bracket Theorem** justification'ıyla LHS → RHS step'ini açar.

## 2f-deep ile karşılaştırma

| Pass | Yöntem | Aksiyom | Adım | Niteliği |
|---|---|---|---|---|
| **2f-theo** (bu) | Theorem cite | 0 | 1 | Paketin sahip olduğu teoremi kullanır |
| **2f-deep** (sonra) | Sıfırdan ispat | 6 | ~100 | Teoremi yeniden ispatlar — Faz 13 pass'i |

**2f-deep**'in motivasyonu pedagojik: "engine teoremi siteden alıp uygulamıyor, terim-terim açıp $[\pi,\pi]_{SN}$'a indiriyor — Derived Bracket Theorem'in *makina seviyesi* ispatı". O pass [plan.md Faz 13](../plan.md) kapsamında ayrı bir alt-projedir.

**2f-theo**'nun motivasyonu pratik: "problem (f)'i bugün, mevcut paket kütüphanesiyle, sıfır ek-aksiyom yazmadan kapatmak". Bu kullanım profilinde paketin değer önerisi — derin teoremlerin tek-satırlık API'a düşmesi — açıkça gösterilir.